# EDA_MODELING Notebook (Researcher 1)

이 노트북은 아래 요구사항을 한 번에 수행하도록 구성되어 있습니다.

1. `train.csv` 기반 EDA + 전처리 + 회귀 모델링
2. RMSE 평가 + `model.pkl` 저장
3. 동일 로직을 `.py`로 정리할 수 있도록 흐름 고정

## 실행 순서 (직접 실행 가이드)
- Step 0: 경로/환경 확인
- Step 1: 데이터 로드 및 기본 점검
- Step 2: 결측치/기술통계/타깃 분포 확인
- Step 3: 피처별 관계 분석(수치형/범주형)
- Step 4: 전처리 + 회귀 모델 학습
- Step 5: RMSE 확인 + 오차 분석
- Step 6: 모델 및 EDA 산출물 저장
- Step 7: 인사이트 마크다운 자동 생성


## 제출/검토 체크리스트
- [ ] RMSE 값 확인
- [ ] `data/shared/model.pkl` 생성 확인
- [ ] `data/shared/metrics.json` 생성 확인
- [ ] `data/shared/eda_summary.md` 생성 확인
- [ ] 노트북 하단 `인사이트 요약` 내용 검토 및 보완


In [24]:
import logging
from pathlib import Path
import json
import pickle
import shutil

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# ── Logger 설정 (노트북 출력용) ──────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S',
    handlers=[logging.StreamHandler()],
    force=True,  # 재실행 시 핸들러 중복 방지
)
logger = logging.getLogger('eda_modeling')

In [25]:
# Step 0) 경로 자동 선택 (Docker / 로컬 환경 자동 분기)
TARGET_COL = 'Performance Index'
RANDOM_STATE = 42

# Docker: /workspace 존재 여부로 분기
if Path('/workspace').exists():
    BASE_DIR = Path('/workspace')
else:
    # 노트북 위치: notebook/modeling/ → 프로젝트 루트는 두 단계 위
    BASE_DIR = Path.cwd().parent.parent

RAW_DIR = BASE_DIR / 'data' / 'raw'
SHARED_DIR = BASE_DIR / 'data' / 'shared'
SHARED_DIR.mkdir(parents=True, exist_ok=True)

train_candidates = [RAW_DIR / 'train.csv', RAW_DIR / 'mission15_train.csv']
test_candidates  = [RAW_DIR / 'test.csv',  RAW_DIR / 'mission15_test.csv']

train_path = next((p for p in train_candidates if p.exists()), None)
test_path  = next((p for p in test_candidates  if p.exists()), None)

if train_path is None:
    raise FileNotFoundError(f'train.csv 또는 mission15_train.csv를 {RAW_DIR} 에 넣어주세요.')

logger.info('BASE_DIR   : %s', BASE_DIR)
logger.info('train_path : %s', train_path)
logger.info('test_path  : %s', test_path if test_path else '(없음)')
logger.info('shared_dir : %s', SHARED_DIR)

17:19:15 [INFO] BASE_DIR   : /Users/youuchul/Documents/github/01_deep_learning/15_Docker


17:19:15 [INFO] train_path : /Users/youuchul/Documents/github/01_deep_learning/15_Docker/data/raw/mission15_train.csv
17:19:15 [INFO] test_path  : /Users/youuchul/Documents/github/01_deep_learning/15_Docker/data/raw/mission15_test.csv
17:19:15 [INFO] shared_dir : /Users/youuchul/Documents/github/01_deep_learning/15_Docker/data/shared


## Step 1) 데이터 로드 및 기본 확인

In [26]:
logger.info('데이터 로드 시작: %s', train_path)
df = pd.read_csv(train_path)
logger.info('데이터 로드 완료 — shape: %s, columns: %s', df.shape, list(df.columns))
display(df.head())

17:19:15 [INFO] 데이터 로드 시작: /Users/youuchul/Documents/github/01_deep_learning/15_Docker/data/raw/mission15_train.csv
17:19:15 [INFO] 데이터 로드 완료 — shape: (7000, 6), columns: ['Hours Studied', 'Previous Scores', 'Extracurricular Activities', 'Sleep Hours', 'Sample Question Papers Practiced', 'Performance Index']


,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,6,73,No,7,2,58.0
1,1,89,Yes,7,2,64.0
2,3,97,Yes,8,0,75.0
3,8,70,No,5,5,59.0
4,7,94,Yes,7,4,86.0


In [27]:
if TARGET_COL not in df.columns:
    raise ValueError(f'타깃 컬럼 {TARGET_COL!r} 이(가) 없습니다.')

missing = df.isna().sum()
missing_cols = missing[missing > 0]
if missing_cols.empty:
    logger.info('결측치 없음')
else:
    logger.warning('결측치 있는 컬럼:\n%s', missing_cols.to_string())

display(df.dtypes.to_frame('dtype'))
display(missing.to_frame('missing_count'))

17:19:15 [INFO] 결측치 없음


,dtype
Hours Studied,int64
Previous Scores,int64
Extracurricular Activities,object
Sleep Hours,int64
Sample Question Papers Practiced,int64
Performance Index,float64


,missing_count
Hours Studied,0
Previous Scores,0
Extracurricular Activities,0
Sleep Hours,0
Sample Question Papers Practiced,0
Performance Index,0


## Step 2) EDA - 기술통계 / 분포 / 상관관계

In [28]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in df.columns if c not in numeric_cols]

logger.info('수치형 컬럼 (%d개): %s', len(numeric_cols), numeric_cols)
logger.info('범주형 컬럼 (%d개): %s', len(categorical_cols), categorical_cols)

display(df[numeric_cols].describe().T)

17:19:15 [INFO] 수치형 컬럼 (5개): ['Hours Studied', 'Previous Scores', 'Sleep Hours', 'Sample Question Papers Practiced', 'Performance Index']
17:19:15 [INFO] 범주형 컬럼 (1개): ['Extracurricular Activities']


,count,mean,std,min,25%,50%,75%,max
Hours Studied,7000.0,4.950000,2.590621,1.0,3.0,5.0,7.0,9.0
Previous Scores,7000.0,69.429714,17.289197,40.0,54.0,69.0,85.0,99.0
Sleep Hours,7000.0,6.530571,1.696144,4.0,5.0,7.0,8.0,9.0
Sample Question Papers Practiced,7000.0,4.607429,2.863550,0.0,2.0,5.0,7.0,9.0
Performance Index,7000.0,55.095143,19.151574,10.0,40.0,55.0,70.0,100.0


In [29]:
# 타깃 분포 히스토그램 + 평균선
mean_val = df[TARGET_COL].mean()
fig_target = px.histogram(
    df, x=TARGET_COL, nbins=30,
    title=f'타깃 분포: {TARGET_COL}',
    template='plotly_white',
    color_discrete_sequence=['steelblue'],
    labels={TARGET_COL: '성취도 지수'},
)
fig_target.add_vline(
    x=mean_val, line_dash='dash', line_color='red',
    annotation_text=f'평균 {mean_val:.1f}', annotation_position='top right'
)
fig_target.update_layout(bargap=0.05)
fig_target.show()

# 상관관계 히트맵 (색상 스케일 개선: -1 ~ +1 고정, diverging 컬러맵)
if len(numeric_cols) > 1:
    corr = df[numeric_cols].corr(numeric_only=True)
    fig_corr = px.imshow(
        corr,
        text_auto='.2f',
        title='수치형 피처 상관관계 히트맵',
        color_continuous_scale='RdBu_r',
        zmin=-1, zmax=1,
        template='plotly_white',
    )
    fig_corr.update_layout(width=620, height=520)
    fig_corr.show()

## EDA 분석 결과 정리

### 타깃 변수 (Performance Index)
- **범위**: 10 ~ 100, **평균**: ~55, **표준편차**: ~19
- 분포는 roughly uniform에 가까우며, 특정 점수대에 쏠림 없음

### 수치형 피처 기술통계 요약
| 피처 | 범위 | 평균 |
|---|---|---|
| Hours Studied | 1 ~ 9 | 4.95 |
| Previous Scores | 40 ~ 99 | 69.4 |
| Sleep Hours | 4 ~ 9 | 6.5 |
| Sample Question Papers Practiced | 0 ~ 9 | 4.6 |

### 상관관계 포인트
- **Previous Scores**와 타깃의 상관이 가장 높을 것으로 예상 (학업 연속성)
- **Hours Studied**도 양의 상관 예상
- **Sleep Hours**, **Sample Question Papers**는 상대적으로 낮은 상관
- 수치형 피처들 간 상호 상관은 낮은 편 → 다중공선성 문제 낮음

### 범주형 피처
- **Extracurricular Activities** (Yes/No): 그룹 간 Performance Index 평균 차이 확인 필요
  - 참여/비참여에 따른 점수 분포를 박스플롯으로 비교

## Step 3) 피처별 관계 분석

In [30]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# 4개 scatter를 2x2 서브플롯으로 한 화면에 비교
feat_cols_scatter = [c for c in numeric_cols if c != TARGET_COL]
n_cols_sp = 2
n_rows_sp = (len(feat_cols_scatter) + 1) // 2

fig_scatter = make_subplots(
    rows=n_rows_sp, cols=n_cols_sp,
    subplot_titles=feat_cols_scatter,
    horizontal_spacing=0.12,
    vertical_spacing=0.18,
)

for i, col in enumerate(feat_cols_scatter):
    r = i // n_cols_sp + 1
    c = i % n_cols_sp + 1

    # 산점도
    fig_scatter.add_trace(
        go.Scatter(
            x=df[col], y=df[TARGET_COL],
            mode='markers',
            marker=dict(size=3, opacity=0.3, color='steelblue'),
            name=col, showlegend=False,
        ),
        row=r, col=c,
    )

    # OLS 추세선 (np.polyfit 사용 — statsmodels 없이도 동작)
    m, b = np.polyfit(df[col], df[TARGET_COL], 1)
    x_line = np.array([df[col].min(), df[col].max()])
    y_line = m * x_line + b
    fig_scatter.add_trace(
        go.Scatter(
            x=x_line, y=y_line,
            mode='lines',
            line=dict(color='red', width=2),
            name=f'trend_{col}', showlegend=False,
        ),
        row=r, col=c,
    )

    # 상관계수 annotation
    corr_val = df[[col, TARGET_COL]].corr().iloc[0, 1]
    fig_scatter.add_annotation(
        text=f'r = {corr_val:.3f}',
        xref=f'x{i+1}', yref=f'y{i+1}',
        x=df[col].max(), y=df[TARGET_COL].min(),
        showarrow=False,
        font=dict(size=12, color='darkred'),
        bgcolor='rgba(255,255,255,0.7)',
        row=r, col=c,
    )

fig_scatter.update_layout(
    title_text=f'수치형 피처 vs {TARGET_COL} (빨간선: OLS 추세, r: 피어슨 상관)',
    height=420 * n_rows_sp,
    template='plotly_white',
)
fig_scatter.show()

In [31]:
for col in categorical_cols:
    grp_stats = df.groupby(col)[TARGET_COL].agg(['count', 'mean', 'std']).round(3)
    fig = px.box(
        df,
        x=col,
        y=TARGET_COL,
        color=col,
        points='outliers',   # 'all' → 7000점으로 너무 느림. 이상치만 표시
        title=f'{col} 그룹별 {TARGET_COL} 분포',
        template='plotly_white',
        labels={TARGET_COL: '성취도 지수', col: col},
    )
    # 그룹별 평균 표시
    for group_name, row_data in grp_stats.iterrows():
        fig.add_hline(
            y=row_data['mean'],
            line_dash='dot', line_color='gray',
            annotation_text=f'{group_name} 평균 {row_data["mean"]:.1f}',
            annotation_position='right',
        )
    fig.show()
    display(grp_stats)

,count,mean,std
Extracurricular Activities,,,
No,3522,54.614,19.122
Yes,3478,55.583,19.172


## 다중공선성(Multicollinearity) 분석

선형 회귀 전 피처 간 다중공선성 여부 확인 (VIF 기준)

- **VIF < 5**: 문제 없음
- **5 ≤ VIF < 10**: 주의
- **VIF ≥ 10**: 심각한 다중공선성 → 피처 제거 또는 Ridge 회귀 고려

In [32]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# num_features는 Step 4에서 정의되므로 여기서 직접 계산 (타깃 제외 수치형)
vif_cols = [c for c in numeric_cols if c != TARGET_COL]
X_vif = df[vif_cols].copy()
X_vif_const = sm.add_constant(X_vif)

vif_data = pd.DataFrame({
    'feature': vif_cols,
    'VIF': [
        variance_inflation_factor(X_vif_const.values, i + 1)
        for i in range(len(vif_cols))
    ]
}).sort_values('VIF', ascending=False)

logger.info('VIF 분석 완료')
display(vif_data)

fig_vif = px.bar(
    vif_data,
    x='feature', y='VIF',
    title='Variance Inflation Factor (VIF) — 다중공선성 진단',
    color='VIF',
    color_continuous_scale='RdYlGn_r',
    text='VIF',
    template='plotly_white',
)
fig_vif.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig_vif.add_hline(y=5,  line_dash='dash', line_color='orange', annotation_text='VIF=5 (주의 기준)')
fig_vif.add_hline(y=10, line_dash='dash', line_color='red',    annotation_text='VIF=10 (위험 기준)')
fig_vif.update_layout(yaxis_range=[0, max(vif_data['VIF'].max() * 1.3, 12)])
fig_vif.show()

17:19:16 [INFO] VIF 분석 완료


,feature,VIF
3,Sample Question Papers Practiced,1.000612
1,Previous Scores,1.000494
0,Hours Studied,1.000488
2,Sleep Hours,1.000320


## 다중공선성 분석 결과 해석

VIF 값이 모두 **5 미만**이면 다중공선성 문제 없음 → 선형 회귀 적용에 적합

> 결과에서 특정 피처가 VIF ≥ 5인 경우:
> - 해당 피처와 강하게 상관된 다른 피처 확인
> - Ridge/Lasso 회귀로 전환 또는 피처 제거 고려

**이 데이터의 수치형 피처들은 서로 독립적으로 설계된 변수들이므로 VIF가 낮을 것으로 예상됨**

## Step 4) 전처리 + 모델 학습 (scikit-learn Pipeline)

In [33]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

cat_features = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
num_features = [c for c in X.columns if c not in cat_features]

logger.info('피처 구성 — 수치형: %s, 범주형: %s', num_features, cat_features)

preprocess = ColumnTransformer(
    transformers=[
        ('categorical', OneHotEncoder(handle_unknown='ignore'), cat_features),
        ('numeric', 'passthrough', num_features),
    ],
    remainder='drop',
)

model = Pipeline(
    steps=[
        ('preprocess', preprocess),
        ('regressor', LinearRegression()),
    ]
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
logger.info('train/valid 분할 — train: %d rows, valid: %d rows', len(X_train), len(X_valid))

logger.info('모델 학습 시작 (LinearRegression + Pipeline)')
model.fit(X_train, y_train)
pred_valid = model.predict(X_valid)
logger.info('모델 학습 완료')

17:19:17 [INFO] 피처 구성 — 수치형: ['Hours Studied', 'Previous Scores', 'Sleep Hours', 'Sample Question Papers Practiced'], 범주형: ['Extracurricular Activities']
17:19:17 [INFO] train/valid 분할 — train: 5600 rows, valid: 1400 rows
17:19:17 [INFO] 모델 학습 시작 (LinearRegression + Pipeline)
17:19:17 [INFO] 모델 학습 완료


## Step 5) 평가 (RMSE 중심)

In [34]:
rmse = root_mean_squared_error(y_valid, pred_valid)
mae  = mean_absolute_error(y_valid, pred_valid)
r2   = r2_score(y_valid, pred_valid)

metrics = {
    'rmse': float(round(rmse, 6)),
    'mae':  float(round(mae, 6)),
    'r2':   float(round(r2, 6)),
    'train_rows': int(X_train.shape[0]),
    'valid_rows': int(X_valid.shape[0]),
    'features': X.columns.tolist(),
}

logger.info('평가 결과 — RMSE: %.4f | MAE: %.4f | R²: %.4f', rmse, mae, r2)

display(pd.DataFrame([metrics]))

17:19:17 [INFO] 평가 결과 — RMSE: 2.0103 | MAE: 1.5912 | R²: 0.9893


,rmse,mae,r2,train_rows,valid_rows,features
0,2.010259,1.591205,0.989281,5600,1400,"[Hours Studied, Previous Scores, Extracurricul..."


In [35]:
residual_df = pd.DataFrame({
    'actual': y_valid.values,
    'pred': pred_valid,
    'residual': y_valid.values - pred_valid,
    'abs_residual': np.abs(y_valid.values - pred_valid),
})

fig_res = px.scatter(
    residual_df,
    x='pred', y='residual',
    color='abs_residual',
    color_continuous_scale='RdYlGn_r',
    opacity=0.5,
    title='잔차 플롯 (Predicted vs Residual)<br><sup>잔차가 y=0 주변에 무작위 분포하면 선형 회귀 가정 충족</sup>',
    labels={'pred': '예측값', 'residual': '잔차 (실제-예측)', 'abs_residual': '|잔차|'},
    template='plotly_white',
)
# y=0 기준선 — 잔차가 이 선 주변에 고루 퍼져야 정상
fig_res.add_hline(
    y=0, line_dash='dash', line_color='red',
    annotation_text='잔차=0 (이상적인 기준선)',
    annotation_position='bottom right',
)
fig_res.show()

display(residual_df[['residual', 'abs_residual']].describe().T.round(4))

,count,mean,std,min,25%,50%,75%,max
residual,1400.0,-0.0083,2.0110,-7.0658,-1.3817,-0.1422,1.2728,7.513
abs_residual,1400.0,1.5912,1.2289,0.0010,0.5942,1.3493,2.2447,7.513


## Step 6) 모델/메트릭/EDA 요약 저장

In [36]:
model_path    = SHARED_DIR / 'model.pkl'
metrics_path  = SHARED_DIR / 'metrics.json'
eda_json_path = SHARED_DIR / 'eda_summary.json'
eda_md_path   = SHARED_DIR / 'eda_summary.md'

with model_path.open('wb') as f:
    pickle.dump(model, f)
logger.info('모델 저장: %s', model_path)

metrics_path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
logger.info('메트릭 저장: %s', metrics_path)

eda_summary = {
    'row_count':    int(df.shape[0]),
    'column_count': int(df.shape[1]),
    'columns':      df.columns.tolist(),
    'dtypes':       {k: str(v) for k, v in df.dtypes.items()},
    'missing_values': df.isna().sum().to_dict(),
    'target_stats': df[TARGET_COL].describe().round(4).to_dict(),
}

eda_json_path.write_text(json.dumps(eda_summary, ensure_ascii=False, indent=2), encoding='utf-8')

eda_lines = [
    '# EDA Summary', '',
    f'- Rows: {eda_summary["row_count"]}',
    f'- Columns: {eda_summary["column_count"]}', '',
    '## Missing Values',
]
for k, v in eda_summary['missing_values'].items():
    eda_lines.append(f'- {k}: {v}')
eda_lines.extend(['', '## Target Stats'])
for k, v in eda_summary['target_stats'].items():
    eda_lines.append(f'- {k}: {v}')

eda_md_path.write_text('\n'.join(eda_lines), encoding='utf-8')
logger.info('EDA 요약 저장: %s, %s', eda_json_path, eda_md_path)

if test_path is not None and test_path.exists():
    copied_test_path = SHARED_DIR / test_path.name
    shutil.copy2(test_path, copied_test_path)
    logger.info('test 파일 복사: %s', copied_test_path)
else:
    logger.warning('test 파일 없음 — 복사 생략')

17:19:17 [INFO] 모델 저장: /Users/youuchul/Documents/github/01_deep_learning/15_Docker/data/shared/model.pkl
17:19:17 [INFO] 메트릭 저장: /Users/youuchul/Documents/github/01_deep_learning/15_Docker/data/shared/metrics.json
17:19:17 [INFO] EDA 요약 저장: /Users/youuchul/Documents/github/01_deep_learning/15_Docker/data/shared/eda_summary.json, /Users/youuchul/Documents/github/01_deep_learning/15_Docker/data/shared/eda_summary.md
17:19:17 [INFO] test 파일 복사: /Users/youuchul/Documents/github/01_deep_learning/15_Docker/data/shared/mission15_test.csv


## Step 7) 인사이트 요약 (자동 생성 + 수동 보완)
아래 셀 실행 후 출력된 내용을 보고, 필요한 문장을 직접 보완해서 보고서/발표에 사용하세요.

In [37]:
target_desc = df[TARGET_COL].describe()
corr_target = df[numeric_cols].corr(numeric_only=True)[TARGET_COL].drop(TARGET_COL).sort_values(key=np.abs, ascending=False)
top_corr = corr_target.head(3)

insight_lines = [
    '### 인사이트 요약 (실행 결과 기반)',
    f'- 데이터 크기: {df.shape[0]} rows x {df.shape[1]} cols',
    f'- 타깃 평균/표준편차: {target_desc["mean"]:.3f} / {target_desc["std"]:.3f}',
    f'- 모델 성능(RMSE): {metrics["rmse"]}',
    f'- 보조 지표(MAE, R2): {metrics["mae"]}, {metrics["r2"]}',
    '- 타깃과 절대 상관이 큰 수치형 피처 Top3:',
]

for col, val in top_corr.items():
    insight_lines.append(f'  - {col}: corr={val:.3f}')

if categorical_cols:
    insight_lines.append('- 범주형 피처 그룹별 평균 비교 결과도 박스플롯과 groupby 표에서 확인 필요')

insight_md = '\n'.join(insight_lines)
display(Markdown(insight_md))

insights_path = SHARED_DIR / 'insights_from_notebook.md'
insights_path.write_text(insight_md, encoding='utf-8')
logger.info('인사이트 저장: %s', insights_path)

### 인사이트 요약 (실행 결과 기반)
- 데이터 크기: 7000 rows x 6 cols
- 타깃 평균/표준편차: 55.095 / 19.152
- 모델 성능(RMSE): 2.010259
- 보조 지표(MAE, R2): 1.591205, 0.989281
- 타깃과 절대 상관이 큰 수치형 피처 Top3:
  - Previous Scores: corr=0.914
  - Hours Studied: corr=0.374
  - Sample Question Papers Practiced: corr=0.050
- 범주형 피처 그룹별 평균 비교 결과도 박스플롯과 groupby 표에서 확인 필요

17:19:17 [INFO] 인사이트 저장: /Users/youuchul/Documents/github/01_deep_learning/15_Docker/data/shared/insights_from_notebook.md


## 다음 단계
- 이 노트북 결과가 만족스러우면 동일 로직을 `src/modeling/train_model.py`에서 실행해 Docker 파이프라인에 연결
- 이후 `src/inference/run_inference.py`로 `result.csv` 생성 검증